In [ ]:
# %%
# ============================================================================
# GNN (GATv2) Nutrient-Aware Pathogen Suppression Predictor  (Run03, VERSION 3)
#
# Same dataset, same splits, same evaluation protocol as the V3 tree notebooks.
#
# WHAT THE GRAPH IS -- read this before anything else.
#
# Your earlier GAT-SS work put SPECIES on the nodes. Version 3 has no species
# columns at all, so that graph does not exist here. This script uses a
# different and, for V3, the only sensible construction:
#
#     nodes  = the carbon sources that are ACTIVE in this broth (1 to 16 of them)
#     node features = that nutrient's concentration R0_cN, plus a learned
#                     embedding of WHICH nutrient it is
#     edges  = complete graph among the active nutrients (every co-present pair)
#     target = community_suppression for the whole graph
#
# So this is a nutrient-interaction graph, not a species-interaction graph. The
# model learns one shared function for "how does nutrient A modify the effect of
# co-present nutrient B", applied to every pair, instead of learning 40
# independent input weights the way the MLP does.
#
# Three properties this buys you over the MLP:
#   1. Variable diversity is native. A D=2 broth is a 2-node graph and a D=16
#      broth is a 16-node graph; nothing is zero-padded.
#   2. Permutation equivariance across nutrients, so the pair-interaction
#      function is shared rather than relearned for each of the 190 pairs.
#   3. The attention weights are directly readable as "which co-present nutrient
#      does the model look at", which the MLP cannot give you.
#
# And the honest caveat: with only 20 possible nutrients, this is close to an
# attention-based set encoder. It is not guaranteed to beat the MLP or the trees,
# and the point of running it is to find out.
# ============================================================================

VERSION = 3

import os
import json
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_add_pool, global_max_pool

from sklearn.model_selection import train_test_split, GroupKFold, GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

DATA_PATH = os.environ.get(
    "SUPPRESSION_CSV",
    r"/Users/PrathamShah/Desktop/Tubingen /Codes/Final_Dataset/community_index_final.csv"
)
OUTDIR = os.environ.get("/Users/PrathamShah/Desktop/Tubingen /Codes/GNN", ".")
os.makedirs(OUTDIR, exist_ok=True)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GSS_SPLITS = int(os.environ.get("GSS_SPLITS", 5))    # GNN fits are the slowest; raise on GPU
MAX_EPOCHS = int(os.environ.get("MAX_EPOCHS", 120))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 256))

print(f"Device: {DEVICE}   torch {torch.__version__}")


def set_seed(seed=SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()


# %%
# ---------------------------------------------------------------- load data
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nTarget (community_suppression) statistics:")
print(df['community_suppression'].describe().round(4))
print("\nRows per diversity level:")
print(df['diversity_level'].value_counts().sort_index())


# %%
# --------------------------------------------------------- feature columns
r0_cols = [f"R0_c{k}" for k in range(1, 21)]
active_cols = [f"active_c{k}" for k in range(1, 21)]
feature_cols = r0_cols + active_cols
version_label = "Pure nutrients only (no species, no size)"
N_NUTRIENTS = len(r0_cols)

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found. First 3: {missing[:3]}")

X = df[feature_cols]
y = df['community_suppression']
groups = df["broth_id"]

print(f"\nVersion: {version_label}")
print(f"Nutrients (potential nodes): {N_NUTRIENTS}")
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")


# %%
# --------------------------------------------------- split + inner holdout
strat_key = df["community_size"].astype(str) + "_" + df["diversity_level"].astype(str)
idx_all = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx_all, test_size=0.20, random_state=42, stratify=strat_key)
# Splitting POSITIONAL INDICES rather than the DataFrame itself, because the
# graph list built below is a plain Python list and has to be indexed by
# position. Keeping one index space for the whole script avoids the classic
# mismatch between .loc labels and list positions.

groups_np = groups.values
print(f"\nTrain: {len(idx_train):,}   Test: {len(idx_test):,}")


def inner_holdout_idx(idx, frac=0.15, seed=42):
    """Group-aware inner split, returning positional indices into the full dataset."""
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    a, b = next(gss.split(idx, groups=groups_np[idx]))
    return idx[a], idx[b]


# %%
# ==================== GRAPH CONSTRUCTION ====================================

R0_MAT = df[r0_cols].values.astype(np.float32)
ACT_MAT = df[active_cols].values.astype(np.int64)
Y_VEC = y.values.astype(np.float32)

# Normalisation constants come from TRAINING ROWS ONLY, over active entries.
# Computing them over all rows would leak the test set's concentration
# distribution into the preprocessing, and computing them over zeros too would
# be dominated by the absent nutrients rather than the present ones.
_train_active_vals = R0_MAT[idx_train][ACT_MAT[idx_train] > 0]
R0_MEAN = float(_train_active_vals.mean())
R0_STD = float(_train_active_vals.std() + 1e-8)
print(f"R0 normalisation from training actives: mean={R0_MEAN:.4f}  std={R0_STD:.4f}")


def build_graph(i):
    """Turn dataset row i into one PyG Data object.

    A Data object is just a container of tensors with a few reserved names.
    The three that matter here:

      x           node feature matrix, shape [num_nodes, num_node_features]
      edge_index  connectivity in COO format, shape [2, num_edges], where
                  column k is the directed edge edge_index[0,k] -> edge_index[1,k].
                  Undirected edges must appear TWICE, once in each direction;
                  PyG does not symmetrise for you.
      y           the graph-level target

    Anything else you attach travels with the object and is batched too, which
    is how `nid` (the nutrient identity for the embedding table) rides along.
    """
    act = np.flatnonzero(ACT_MAT[i] > 0)
    if len(act) == 0:                       # defensive: a broth with nothing in it
        act = np.array([0])

    conc = (R0_MAT[i, act] - R0_MEAN) / R0_STD

    n = len(act)
    if n > 1:
        src, dst = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
        mask = src != dst                   # drop self-loops; the layer adds them back
        edge_index = np.stack([src[mask], dst[mask]])
    else:
        edge_index = np.zeros((2, 0), dtype=np.int64)
    # A complete graph among co-present nutrients encodes "these were in the
    # broth together" and nothing more. That is deliberate: we do not know the
    # true chemical interaction network, so imposing a sparser structure would be
    # inventing biology. Attention is what learns which of those pairs matter.
    # Once the chemistry-grounded features land, a sparser chemically motivated
    # graph becomes the obvious thing to try instead.

    return Data(
        x=torch.tensor(conc, dtype=torch.float32).unsqueeze(1),   # [n, 1]
        nid=torch.tensor(act, dtype=torch.long),                  # [n] nutrient identity
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        y=torch.tensor([Y_VEC[i]], dtype=torch.float32),
    )


print("\nBuilding graphs...")
t0 = time.time()
graphs = [build_graph(i) for i in range(len(df))]
print(f"Built {len(graphs):,} graphs in {time.time() - t0:.1f}s")

_n_nodes = np.array([g.num_nodes for g in graphs])
_n_edges = np.array([g.edge_index.shape[1] for g in graphs])
print(f"Nodes per graph:  min={_n_nodes.min()}  median={np.median(_n_nodes):.0f}  "
      f"max={_n_nodes.max()}")
print(f"Edges per graph:  min={_n_edges.min()}  median={np.median(_n_edges):.0f}  "
      f"max={_n_edges.max()}")
print("Example graph (row 0):", graphs[0])


# %%
# ==================== THE MODEL =============================================

class NutrientGAT(nn.Module):
    """Graph attention network over the co-present-nutrient graph."""

    def __init__(self, n_nutrients=N_NUTRIENTS, emb_dim=16, hidden=64, heads=4,
                 n_layers=2, dropout=0.15, readout_hidden=64):
        super().__init__()

        self.embedding = nn.Embedding(n_nutrients, emb_dim)
        # This is the piece that makes the graph formulation legitimate.
        # A GNN is permutation-equivariant: it cannot tell node 3 from node 17
        # except through their features. Nutrients are NOT interchangeable --
        # your SHAP results say R0_c8 and R0_c6 dominate -- so without an
        # identity signal the model would be strictly WEAKER than the MLP,
        # having thrown away which nutrient is which. nn.Embedding is a lookup
        # table of `n_nutrients` learned vectors of length `emb_dim`, trained
        # jointly with everything else, giving each carbon source a learned
        # chemical fingerprint. When the RDKit descriptors arrive, this is
        # exactly the slot they replace.

        self.conc_proj = nn.Linear(1, emb_dim)
        in_dim = emb_dim * 2   # identity embedding concatenated with concentration

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for layer in range(n_layers):
            self.convs.append(GATv2Conv(
                in_channels=in_dim if layer == 0 else hidden * heads,
                out_channels=hidden,
                heads=heads,
                concat=True,
                dropout=dropout,
                add_self_loops=True,
            ))
            self.norms.append(nn.LayerNorm(hidden * heads))
        # GATv2Conv, not GATConv. In the original GAT the attention logit is
        # a^T [Wh_i || Wh_j] with the LeakyReLU applied afterwards, which makes
        # the attention "static": the ranking of which neighbour is most
        # interesting turns out to be the same for every query node, only the
        # sharpness changes. GATv2 moves the nonlinearity inside,
        # a^T LeakyReLU(W[h_i || h_j]), which restores dynamic attention. For
        # this problem that matters directly -- you want nutrient 8 to be the
        # most attended neighbour in some broths and not in others.
        #
        # heads=4 with concat=True runs four independent attention mechanisms
        # and concatenates them, so the output is hidden*heads wide. Multi-head
        # attention stabilises training: one head latching onto a spurious
        # pattern does not dominate. Set concat=False to average them instead,
        # which keeps the width at `hidden`.
        #
        # dropout inside GATv2Conv drops ATTENTION COEFFICIENTS, not features.
        # It stops the model relying on one specific edge.
        #
        # add_self_loops=True is essential here, not cosmetic: a D=1 broth is a
        # single node with zero edges, and without a self-loop it would receive
        # no messages at all and its embedding would be whatever the bias term
        # says. With self-loops every node always attends to itself.
        #
        # LayerNorm rather than BatchNorm between layers. Graphs in a batch have
        # different node counts, so batch statistics are computed over a varying
        # and unbalanced population; LayerNorm normalises each node's own feature
        # vector and is unaffected by how the batch happened to be composed.

        self.dropout = dropout

        self.readout = nn.Sequential(
            nn.Linear(hidden * heads * 3, readout_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(readout_hidden, 1),
        )
        # Input is hidden*heads*3 because three pooling operations are
        # concatenated. Each answers a different question and they are not
        # redundant:
        #   global_mean_pool -- what is the average character of this broth
        #   global_add_pool  -- total dose; scales with how many nutrients are
        #                       present, so it is the channel that carries
        #                       diversity level and total carbon
        #   global_max_pool  -- is any single nutrient extreme; captures a
        #                       dominant carbon source that a mean would dilute
        # Mean alone would make a 1-nutrient and a 16-nutrient broth of the same
        # average composition look identical, which is exactly the distinction
        # you need.

    def forward(self, data, return_attention=False):
        x, nid, edge_index, batch = data.x, data.nid, data.edge_index, data.batch

        h = torch.cat([self.embedding(nid), self.conc_proj(x)], dim=-1)

        attentions = []
        for conv, norm in zip(self.convs, self.norms):
            if return_attention:
                h, att = conv(h, edge_index, return_attention_weights=True)
                attentions.append(att)
                # With return_attention_weights=True the layer returns
                # (node_embeddings, (edge_index_with_self_loops, alpha)) where
                # alpha has shape [num_edges, heads]. Note the returned
                # edge_index includes the added self-loops, so it is LONGER than
                # the one you passed in -- indexing the original edge list with
                # these positions would silently misalign.
            else:
                h = conv(h, edge_index)
            h = norm(h)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            # F.dropout needs training=self.training passed explicitly. The
            # functional form has no idea whether the module is in train or eval
            # mode, so omitting this leaves dropout active during evaluation and
            # produces quietly noisy predictions.

        pooled = torch.cat([
            global_mean_pool(h, batch),
            global_add_pool(h, batch),
            global_max_pool(h, batch),
        ], dim=-1)
        # `batch` is a vector of length num_nodes assigning each node to its
        # graph in the batch. PyG batches by stacking graphs into one big
        # block-diagonal graph with node indices offset, so the pooling
        # functions use `batch` as a scatter index to produce one row per graph.

        return (self.readout(pooled).squeeze(-1), attentions) if return_attention \
            else self.readout(pooled).squeeze(-1)


# %%
# ==================== TRAINING LOOP =========================================

def loaders_from_idx(fit_idx, val_idx, batch_size=BATCH_SIZE):
    fit_loader = DataLoader([graphs[i] for i in fit_idx], batch_size=batch_size,
                            shuffle=True)
    val_loader = DataLoader([graphs[i] for i in val_idx], batch_size=batch_size * 4,
                            shuffle=False)
    return fit_loader, val_loader
    # PyG's DataLoader is not torch's: it uses a collate function that merges a
    # list of Data objects into one disconnected batched graph and builds the
    # `batch` vector. Using torch.utils.data.DataLoader here would try to stack
    # tensors of different shapes and fail.


@torch.no_grad()
def predict_graphs(model, idx, batch_size=BATCH_SIZE * 4, clip=(0.0, 1.0)):
    model.eval()
    loader = DataLoader([graphs[i] for i in idx], batch_size=batch_size, shuffle=False)
    out = [model(b.to(DEVICE)).cpu().numpy() for b in loader]
    pred = np.concatenate(out)
    return np.clip(pred, *clip) if clip is not None else pred


def train_gnn(fit_idx, val_idx, params, max_epochs=MAX_EPOCHS, patience=15,
              seed=SEED, verbose=False, record=False):
    """Fit one GNN with early stopping on a group-aware validation set."""
    set_seed(seed)

    model = NutrientGAT(
        emb_dim=params.get('emb_dim', 16),
        hidden=params.get('hidden', 64),
        heads=params.get('heads', 4),
        n_layers=params.get('n_layers', 2),
        dropout=params.get('dropout', 0.15),
        readout_hidden=params.get('readout_hidden', 64),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params.get('lr', 3e-3),
                                  weight_decay=params.get('weight_decay', 1e-4))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=max(3, patience // 3))
    criterion = nn.MSELoss()

    fit_loader, val_loader = loaders_from_idx(fit_idx, val_idx,
                                              params.get('batch_size', BATCH_SIZE))

    best_val, best_state, best_epoch, bad = np.inf, None, 0, 0
    hist = {'train': [], 'valid': []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        running, n_seen = 0.0, 0
        for b in fit_loader:
            b = b.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            running += loss.item() * b.num_graphs
            n_seen += b.num_graphs
            # b.num_graphs, not len(b): a batched Data object's length is its
            # number of attributes, while num_graphs is how many graphs were
            # merged. Using the wrong one silently mis-weights the running loss.

        tr_loss = running / n_seen

        model.eval()
        vs, vn = 0.0, 0
        with torch.no_grad():
            for b in val_loader:
                b = b.to(DEVICE)
                vs += float(((model(b) - b.y) ** 2).sum())
                vn += b.num_graphs
        val_loss = vs / vn

        hist['train'].append(tr_loss)
        hist['valid'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val - 1e-6:
            best_val, best_epoch, bad = val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"    epoch {epoch:>3}  train MSE={tr_loss:.5f}  val MSE={val_loss:.5f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    model.load_state_dict(best_state)
    return (model, best_epoch, best_val, hist) if record else (model, best_epoch, best_val)


# %%
# ------------------------------------------------------------ initial model
fit_idx, val_idx = inner_holdout_idx(idx_train)
print(f"\nInner split: fit={len(fit_idx):,}  val={len(val_idx):,}  "
      f"(broths disjoint: {len(set(groups_np[fit_idx]) & set(groups_np[val_idx])) == 0})")

t0 = time.time()
m_init, ep_init, val_init = train_gnn(fit_idx, val_idx, params={},
                                      max_epochs=min(MAX_EPOCHS, 40),
                                      patience=10, verbose=True)
yp_init = predict_graphs(m_init, idx_test)
print(f"\nGNN (initial):  R2={r2_score(Y_VEC[idx_test], yp_init):.4f}  "
      f"MAE={mean_absolute_error(Y_VEC[idx_test], yp_init):.4f}  "
      f"best_epoch={ep_init}  ({time.time() - t0:.1f}s)")


# %%
# ------------------------------------------ hyperparameter tuning (manual)
RUN_TUNING = False

param_grid = [
    dict(emb_dim=16, hidden=64, heads=4, n_layers=2, dropout=0.15, lr=3e-3),
]
# Wider grid for RUN_TUNING = True:
# param_grid = [
#     dict(emb_dim=8,  hidden=32,  heads=2, n_layers=2, dropout=0.10, lr=3e-3),
#     dict(emb_dim=16, hidden=64,  heads=4, n_layers=2, dropout=0.15, lr=3e-3),
#     dict(emb_dim=16, hidden=64,  heads=4, n_layers=3, dropout=0.20, lr=1e-3),
#     dict(emb_dim=32, hidden=128, heads=4, n_layers=2, dropout=0.20, lr=1e-3),
#     dict(emb_dim=32, hidden=128, heads=8, n_layers=3, dropout=0.30, lr=1e-3),
# ]
# On a complete graph, going beyond 2 or 3 layers rarely helps: every node
# already sees every other node after ONE hop, so extra layers add capacity but
# no new receptive field, and they bring oversmoothing -- repeated neighbour
# averaging drives all node embeddings toward the same vector.


def grid_search_gnn(train_idx, grid, n_splits=3):
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    print(f"Searching {len(grid)} candidates x {n_splits} group folds")
    for i, params in enumerate(grid, start=1):
        fold_mse = []
        for a, b in gkf.split(train_idx, groups=groups_np[train_idx]):
            tr, va = train_idx[a], train_idx[b]
            f_i, v_i = inner_holdout_idx(tr, frac=0.15, seed=SEED + i)
            m, _, _ = train_gnn(f_i, v_i, params,
                                max_epochs=min(MAX_EPOCHS, 50), patience=8)
            fold_mse.append(mean_squared_error(Y_VEC[va], predict_graphs(m, va)))
        rows.append({**{k: str(v) for k, v in params.items()},
                     'mean_mse': float(np.mean(fold_mse)),
                     'std_mse': float(np.std(fold_mse))})
        print(f"  [{i}/{len(grid)}] {params} -> MSE {np.mean(fold_mse):.5f}")
    return pd.DataFrame(rows).sort_values('mean_mse').reset_index(drop=True)


if RUN_TUNING:
    tune_df = grid_search_gnn(idx_train, param_grid)
    tune_df.to_csv(os.path.join(OUTDIR, "gnn_tuning.csv"), index=False)
    best_params = param_grid[0]
    print(f"\nTuning table written; using: {best_params}")
else:
    best_params = param_grid[0]
    print(f"Using params (tuning skipped): {best_params}")


# %%
# ------------------------------------------------------------- final model
t0 = time.time()
model_final, best_epoch, best_val, hist = train_gnn(
    fit_idx, val_idx, best_params, verbose=True, record=True)
print(f"Training time: {time.time() - t0:.1f}s   best epoch: {best_epoch}")

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(range(1, len(hist['train']) + 1), hist['train'], color='steelblue', lw=1.5,
           label='Train MSE')
ax_tc.plot(range(1, len(hist['valid']) + 1), hist['valid'], color='darkorange', lw=1.5,
           label='Validation MSE')
ax_tc.axvline(best_epoch, color='red', linestyle='--', lw=1.2, label=f'Best epoch = {best_epoch}')
ax_tc.set_xlabel('Epoch')
ax_tc.set_ylabel('MSE')
ax_tc.set_title(f'GATv2 Training Curve -- best epoch={best_epoch}, val MSE={best_val:.5f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'gnn_training_curve.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: gnn_training_curve.png")

yp_final = predict_graphs(model_final, idx_test)
y_test_np = Y_VEC[idx_test]

r2_final = r2_score(y_test_np, yp_final)
mae_final = mean_absolute_error(y_test_np, yp_final)
rmse_final = float(np.sqrt(mean_squared_error(y_test_np, yp_final)))
n_params = sum(p.numel() for p in model_final.parameters())

print(f"\nGNN final:  R2={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Parameters: {n_params:,}")


# %%
# ------------------------------- GroupKFold: held-out broths + pooled OOF ---
gkf = GroupKFold(n_splits=5)
fold_rows = []
oof_pred = np.full(len(df), np.nan)

print("\nGroupKFold evaluation (held-out broths):")

for fold_num, (a, b) in enumerate(gkf.split(idx_all, groups=groups_np), start=1):
    tr, te = idx_all[a], idx_all[b]
    assert len(set(groups_np[tr]) & set(groups_np[te])) == 0

    f_i, v_i = inner_holdout_idx(tr, frac=0.15, seed=42 + fold_num)
    m_fold, ep_fold, _ = train_gnn(f_i, v_i, best_params, seed=SEED + fold_num)
    yp_fold = predict_graphs(m_fold, te)
    oof_pred[te] = yp_fold

    row = {'fold': fold_num, 'n_test': len(te),
           'r2': r2_score(Y_VEC[te], yp_fold),
           'mae': mean_absolute_error(Y_VEC[te], yp_fold),
           'rmse': float(np.sqrt(mean_squared_error(Y_VEC[te], yp_fold))),
           'pearson': pearsonr(Y_VEC[te], yp_fold).statistic,
           'spearman': spearmanr(Y_VEC[te], yp_fold).statistic,
           'epochs': ep_fold}
    fold_rows.append(row)
    print(f"  Fold {fold_num}: R2={row['r2']:.4f}  MAE={row['mae']:.4f}  "
          f"RMSE={row['rmse']:.4f}  Pearson={row['pearson']:.4f}  "
          f"Spearman={row['spearman']:.4f}  epochs={row['epochs']}")

folds_df = pd.DataFrame(fold_rows)

r2_oof = r2_score(Y_VEC, oof_pred)
mae_oof = mean_absolute_error(Y_VEC, oof_pred)
rmse_oof = float(np.sqrt(mean_squared_error(Y_VEC, oof_pred)))

print(f"\nPOOLED out-of-fold over all {len(df):,} rows:")
print(f"  R2={r2_oof:.4f}  MAE={mae_oof:.4f}  RMSE={rmse_oof:.4f}  "
      f"Spearman={spearmanr(Y_VEC, oof_pred).statistic:.4f}")
print(f"\nInterpolation R2 = {r2_final:.4f}   Held-out-broth R2 = {r2_oof:.4f}   "
      f"Gap = {r2_final - r2_oof:.4f}")


# %%
# ------------------------------------------- GroupShuffleSplit repeatability
gss = GroupShuffleSplit(n_splits=GSS_SPLITS, test_size=0.20, random_state=42)
gss_rows = []

for split_num, (a, b) in enumerate(gss.split(idx_all, groups=groups_np), start=1):
    tr, te = idx_all[a], idx_all[b]
    assert len(set(groups_np[tr]) & set(groups_np[te])) == 0
    f_i, v_i = inner_holdout_idx(tr, frac=0.15, seed=42 + split_num)
    m, ep, _ = train_gnn(f_i, v_i, best_params, seed=SEED + split_num)
    yp = predict_graphs(m, te)
    gss_rows.append({'split': split_num,
                     'r2': r2_score(Y_VEC[te], yp),
                     'mae': mean_absolute_error(Y_VEC[te], yp),
                     'rmse': float(np.sqrt(mean_squared_error(Y_VEC[te], yp))),
                     'epochs': ep})
    print(f"Split {split_num:02d}: R2={gss_rows[-1]['r2']:.4f}  "
          f"MAE={gss_rows[-1]['mae']:.4f}  RMSE={gss_rows[-1]['rmse']:.4f}  epochs={ep}")

gss_df = pd.DataFrame(gss_rows)
print(f"\nGroupShuffleSplit held-out-broth summary ({GSS_SPLITS} splits):")
for c in ['r2', 'mae', 'rmse']:
    print(f"  {c:<6} mean={gss_df[c].mean():.4f}  std={gss_df[c].std():.4f}")


# %%
# ------------------------------------------------ per-condition breakdown
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true': y_test_np,
    'y_pred': yp_final,
    'residual': y_test_np - yp_final,
    'diversity_level': df['diversity_level'].values[idx_test],
    'community_size': df['community_size'].values[idx_test],
})


def condition_metrics(g):
    y_t, y_p = g['y_true'].values, g['y_pred'].values
    return {'n': len(y_t), 'r2': r2_score(y_t, y_p),
            'mae': mean_absolute_error(y_t, y_p),
            'rmse': float(np.sqrt(mean_squared_error(y_t, y_p))),
            'pearson': pearsonr(y_t, y_p).statistic,
            'spearman': spearmanr(y_t, y_p).statistic,
            'bias': float(np.mean(y_p - y_t)), 'std_true': float(np.std(y_t))}


hdr = (f"{'':6} {'R2':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} "
       f"{'Bias':>8} {'std(y)':>7} {'N':>6}")

print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m = condition_metrics(grp)
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['bias']:>8.4f} "
          f"{m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R2 is unreliable here. Read MAE and Spearman instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp)
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['bias']:>8.4f} "
          f"{m['std_true']:>7.4f} {m['n']:>6,}")


# %%
# =============== INTERPRETATION: attention, embeddings, ablation ============
print("\ninterpretation")

# (1) Attention matrix. Aggregate every attention weight in the test set into a
# 20x20 matrix indexed by real nutrient identity, so entry [a, b] reads
# "when nutrient b is updating itself, how much does it look at nutrient a".
model_final.eval()
att_sum = np.zeros((N_NUTRIENTS, N_NUTRIENTS))
att_cnt = np.zeros((N_NUTRIENTS, N_NUTRIENTS))

att_loader = DataLoader([graphs[i] for i in idx_test[:min(5000, len(idx_test))]],
                        batch_size=BATCH_SIZE, shuffle=False)
with torch.no_grad():
    for b in att_loader:
        b = b.to(DEVICE)
        _, atts = model_final(b, return_attention=True)
        ei, alpha = atts[0]                      # first conv layer
        alpha = alpha.mean(dim=1).cpu().numpy()  # average over heads -> [num_edges]
        src = b.nid[ei[0]].cpu().numpy()         # map node index -> nutrient identity
        dst = b.nid[ei[1]].cpu().numpy()
        np.add.at(att_sum, (src, dst), alpha)
        np.add.at(att_cnt, (src, dst), 1.0)
        # np.add.at accumulates in place with repeated indices. Plain
        # att_sum[src, dst] += alpha would apply each duplicated index only
        # ONCE, silently dropping most of the edges.

att_mean = np.divide(att_sum, att_cnt, out=np.zeros_like(att_sum), where=att_cnt > 0)

incoming = att_mean.sum(axis=0)   # total attention each nutrient RECEIVES
print("\nTop 10 nutrients by total attention received:")
for j in np.argsort(-incoming)[:10]:
    print(f"  {r0_cols[j]:<10} {incoming[j]:.4f}")

# (2) Learned nutrient embeddings. Cosine similarity between the 20 learned
# vectors: nutrients the model treats as functionally interchangeable end up
# close together. This is the closest thing to a data-driven chemical grouping
# you can get before the RDKit descriptors exist.
emb = model_final.embedding.weight.detach().cpu().numpy()
emb_n = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
emb_sim = emb_n @ emb_n.T

# (3) Node ablation importance. Drop one nutrient from every graph that contains
# it, and measure the R2 drop. This is the graph analogue of permutation
# importance and is directly comparable to the tree SHAP rankings.
print("\nAblation importance (R2 drop when nutrient removed from its graphs)...")
abl_idx = idx_test[:min(5000, len(idx_test))]
base_r2 = r2_score(Y_VEC[abl_idx], predict_graphs(model_final, abl_idx))
abl = np.zeros(N_NUTRIENTS)

for j in range(N_NUTRIENTS):
    mod = []
    for i in abl_idx:
        g = graphs[i]
        keep = (g.nid != j).nonzero(as_tuple=True)[0]
        if len(keep) == 0:
            keep = torch.arange(g.num_nodes)     # never leave an empty graph
        n = len(keep)
        if n > 1:
            s, d = torch.meshgrid(torch.arange(n), torch.arange(n), indexing='ij')
            mk = s != d
            ei = torch.stack([s[mk], d[mk]])
        else:
            ei = torch.zeros((2, 0), dtype=torch.long)
        mod.append(Data(x=g.x[keep], nid=g.nid[keep], edge_index=ei, y=g.y))
    ld = DataLoader(mod, batch_size=BATCH_SIZE * 4, shuffle=False)
    with torch.no_grad():
        pr = np.concatenate([model_final(bb.to(DEVICE)).cpu().numpy() for bb in ld])
    abl[j] = base_r2 - r2_score(Y_VEC[abl_idx], np.clip(pr, 0, 1))

abl_importance = pd.Series(abl, index=r0_cols).sort_values(ascending=False)
print("\nTop 10 nutrients by ablation importance:")
print(abl_importance.head(10).round(5).to_string())


# %%
# ------------------------------------------------------------- FIGURE panels
output_name = os.path.join(OUTDIR, f"gnn_v{VERSION}_results.png")

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(f'GATv2 on nutrient co-presence graphs -- {version_label}\n'
             f'R2={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
             f'held-out-broth R2={r2_oof:.3f}  params={n_params:,}',
             fontsize=12, fontweight='bold', y=0.99)

divs = sorted(results_df['diversity_level'].unique())
sizes = sorted(results_df['community_size'].unique())

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1, 2, 4, 8, 16], ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test_np[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0, 1], [0, 1], 'r--', lw=1.5)
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R2={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = results_df['residual'].values
for n, col in zip([12, 16, 24, 32, 48], ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5, color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)
top = abl_importance.head(20)
ax3.barh(range(len(top)), top.values[::-1], color='#4CAF50')
ax3.set_yticks(range(len(top)))
ax3.set_yticklabels(top.index[::-1], fontsize=7)
ax3.set_xlabel('R2 drop when nutrient removed')
ax3.set_title('Nutrient Ablation Importance')

ax4 = fig.add_subplot(2, 3, 4)
r2_div = [r2_score(results_df[results_df['diversity_level'] == d]['y_true'],
                   results_df[results_df['diversity_level'] == d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R2={r2_final:.3f}')
ax4.set_xlabel('Diversity level (= number of graph nodes)')
ax4.set_ylabel('R2')
ax4.set_title('R2 by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
r2_sz = [r2_score(results_df[results_df['community_size'] == n]['y_true'],
                  results_df[results_df['community_size'] == n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R2={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R2')
ax5.set_title('R2 by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
ax6.scatter(Y_VEC, oof_pred, alpha=0.10, s=2, c='darkorange', label='Held-out broths (OOF)')
ax6.scatter(y_test_np, yp_final, alpha=0.10, s=2, c='steelblue', label='Interpolation test')
ax6.plot([0, 1], [0, 1], 'r--', lw=1.5)
ax6.set_xlabel('True suppression')
ax6.set_ylabel('Predicted')
ax6.set_title(f'Interpolation vs Held-out Broth\n'
              f'R2 {r2_final:.3f} -> {r2_oof:.3f}  (gap {r2_final - r2_oof:.3f})')
ax6.legend(fontsize=8, markerscale=4)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")


# %%
# ------------------------------------------ FIGURE: graph-specific analysis
graph_name = os.path.join(OUTDIR, f"gnn_v{VERSION}_graph_analysis.png")

fig = plt.figure(figsize=(18, 5.5), dpi=120)
fig.suptitle('GATv2 Graph Analysis -- what the attention and embeddings learned',
             fontsize=12, fontweight='bold', y=1.02)

axA = fig.add_subplot(1, 3, 1)
im = axA.imshow(att_mean, cmap='viridis', aspect='auto')
axA.set_xlabel('Destination nutrient (being updated)')
axA.set_ylabel('Source nutrient (being attended to)')
axA.set_title('Mean Attention Weight by Nutrient Pair')
axA.set_xticks(range(0, N_NUTRIENTS, 2))
axA.set_xticklabels([f'c{k + 1}' for k in range(0, N_NUTRIENTS, 2)], fontsize=7)
axA.set_yticks(range(0, N_NUTRIENTS, 2))
axA.set_yticklabels([f'c{k + 1}' for k in range(0, N_NUTRIENTS, 2)], fontsize=7)
plt.colorbar(im, ax=axA, fraction=0.046)

axB = fig.add_subplot(1, 3, 2)
flat = att_mean[att_cnt > 0]
axB.hist(flat, bins=40, color='#1A6B6B', edgecolor='white')
for d, col in zip([2, 4, 8, 16], ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8']):
    axB.axvline(1.0 / d, color=col, lw=1.5, ls='--', label=f'uniform at D={d}: {1 / d:.3f}')
axB.set_xlabel('Attention weight (averaged over heads)')
axB.set_ylabel('Number of nutrient pairs')
axB.set_title('Are Attention Weights Uniform?\n(peaked away from the lines = real structure)')
axB.legend(fontsize=7)
# If the histogram sits on the uniform lines, GATv2 has learned nothing a simple
# mean aggregator would not have given you, and a GCN or DeepSets baseline would
# do just as well. That is a real and reportable result, not a failure.

axC = fig.add_subplot(1, 3, 3)
im2 = axC.imshow(emb_sim, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axC.set_title('Learned Nutrient Embedding Similarity\n(cosine, red = treated alike)')
axC.set_xticks(range(0, N_NUTRIENTS, 2))
axC.set_xticklabels([f'c{k + 1}' for k in range(0, N_NUTRIENTS, 2)], fontsize=7)
axC.set_yticks(range(0, N_NUTRIENTS, 2))
axC.set_yticklabels([f'c{k + 1}' for k in range(0, N_NUTRIENTS, 2)], fontsize=7)
plt.colorbar(im2, ax=axC, fraction=0.046)

plt.tight_layout()
plt.savefig(graph_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"Plot saved: {graph_name}")


# %%
# ------------------------------------------------- save model and predictions
model_path = os.path.join(OUTDIR, f"gnn_v{VERSION}_model.pt")
torch.save({
    'state_dict': model_final.state_dict(),
    'arch': {k: best_params.get(k) for k in
             ['emb_dim', 'hidden', 'heads', 'n_layers', 'dropout', 'readout_hidden']},
    'r0_mean': R0_MEAN, 'r0_std': R0_STD,
    'r0_cols': r0_cols, 'active_cols': active_cols,
    'n_nutrients': N_NUTRIENTS,
}, model_path)

pd.DataFrame({
    'y_true': y_test_np, 'y_pred': yp_final, 'residual': y_test_np - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size': results_df['community_size'].values,
}).to_csv(os.path.join(OUTDIR, f"gnn_v{VERSION}_predictions.csv"), index=False)

pd.DataFrame({
    'y_true': Y_VEC, 'oof_pred': oof_pred, 'broth_id': groups_np,
    'diversity_level': df['diversity_level'].values,
    'community_size': df['community_size'].values,
}).to_csv(os.path.join(OUTDIR, f"gnn_v{VERSION}_oof_heldout_broth.csv"), index=False)

folds_df.to_csv(os.path.join(OUTDIR, f"gnn_v{VERSION}_groupkfold.csv"), index=False)
gss_df.to_csv(os.path.join(OUTDIR, f"gnn_v{VERSION}_groupshufflesplit.csv"), index=False)
abl_importance.to_csv(os.path.join(OUTDIR, f"gnn_v{VERSION}_ablation_importance.csv"))
pd.DataFrame(att_mean, index=r0_cols, columns=r0_cols).to_csv(
    os.path.join(OUTDIR, f"gnn_v{VERSION}_attention_matrix.csv"))
pd.DataFrame(emb, index=r0_cols).to_csv(
    os.path.join(OUTDIR, f"gnn_v{VERSION}_nutrient_embeddings.csv"))

with open(os.path.join(OUTDIR, f"gnn_v{VERSION}_summary.json"), 'w') as f:
    json.dump({'model': 'gatv2_nutrient_graph', 'version': VERSION,
               'version_label': version_label, 'n_nutrients': N_NUTRIENTS,
               'n_params': int(n_params), 'best_epoch': int(best_epoch),
               'params': {k: str(v) for k, v in best_params.items()},
               'r2_interpolation': float(r2_final), 'mae_interpolation': float(mae_final),
               'rmse_interpolation': float(rmse_final),
               'r2_heldout_broth': float(r2_oof), 'mae_heldout_broth': float(mae_oof),
               'rmse_heldout_broth': float(rmse_oof),
               'gss_r2_mean': float(gss_df['r2'].mean()),
               'gss_r2_std': float(gss_df['r2'].std())}, f, indent=2)

print(f"Model saved: {model_path}")


# %%
# ------------------------------------------------------- reload sanity check
ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
m2 = NutrientGAT(n_nutrients=ckpt['n_nutrients'],
                 **{k: v for k, v in ckpt['arch'].items() if v is not None}).to(DEVICE)
m2.load_state_dict(ckpt['state_dict'])
m2.eval()
chk = DataLoader([graphs[i] for i in idx_test[:5]], batch_size=5, shuffle=False)
with torch.no_grad():
    pr = np.clip(np.concatenate([m2(b.to(DEVICE)).cpu().numpy() for b in chk]), 0, 1)
assert np.allclose(pr, yp_final[:5], atol=1e-5)
print("Reloaded model reproduces predictions exactly.")


# %%
# ------------------------------------------------------------------ summary
print(f"\n{'=' * 68}")
print(f"SUMMARY -- GATv2 nutrient graph VERSION {VERSION}: {version_label}")
print(f"{'=' * 68}")
print(f"Graph:                       nodes = active nutrients, edges = co-presence")
print(f"Nodes per graph:             {_n_nodes.min()} to {_n_nodes.max()}")
print(f"Parameters:                  {n_params:,}")
print(f"Best epoch:                  {best_epoch}")
print(f"Interpolation R2:            {r2_final:.4f}")
print(f"Interpolation MAE:           {mae_final:.4f}")
print(f"Interpolation RMSE:          {rmse_final:.4f}")
print(f"Held-out-broth R2 (pooled):  {r2_oof:.4f}")
print(f"Held-out-broth MAE:          {mae_oof:.4f}")
print(f"Generalization gap (R2):     {r2_final - r2_oof:.4f}")
print(f"GroupShuffleSplit R2:        {gss_df['r2'].mean():.4f} +/- {gss_df['r2'].std():.4f}")
print(f"\nTop 5 nutrients by ablation importance:")
for feat, val in abl_importance.head(5).items():
    print(f"  {feat:<12} {val:.5f}")
print(f"\nTop 5 nutrients by attention received:")
for j in np.argsort(-incoming)[:5]:
    print(f"  {r0_cols[j]:<12} {incoming[j]:.5f}")